# Proje: AlphaZero Mimarisi ile Territory Capture Yapay Zekası

**Amaç:** Bu projenin amacı, AlphaZero algoritmasının temel bileşenlerini kullanarak özel bir masa oyunu (Territory Capture) için yapay zeka geliştirmektir. Projede standart gözetimli öğrenme (Supervised Learning) yerine; Monte Carlo Ağaç Araması (MCTS), Self-Play (Kendi kendine oynama) ve Policy-Value Ağları entegre edilmiştir.

## 1. Ortam Kurulumu ve Donanım Optimizasyonu
Bu bölümde gerekli kütüphaneleri içe aktarıyor ve **NVIDIA A100 GPU**'nun Tensor Çekirdeklerinden (Tensor Cores) maksimum verim almak için **TF32** (TensorFloat-32) donanım optimizasyonlarını aktif ediyoruz.

In [ ]:
import os
import json
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

# Google Drive Bağlantısı
drive.mount('/content/drive')

# A100 GPU Tensor Core Optimizasyonları
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    print("\n[BAŞARILI] NVIDIA A100 GPU bulundu ve TF32 optimizasyonları aktif edildi!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[BAŞARILI] NVIDIA A100 GPU bulundu ve TF32 optimizasyonları aktif edildi!


## 2. Derin Öğrenme Modeli (Policy-Value Network)

Modelimiz bir Evrişimli Sinir Ağıdır (CNN). Oyun tahtasının mevcut durumunu (2x5x5 boyutlarında) alır ve özellikleri çıkardıktan sonra iki ayrı başlığa (head) ayrılır:
1. **Policy Head (İlke Çıktısı):** Ajanın 25 olası hamle üzerinde hangisini seçeceğine dair bir olasılık dağılımı (MCTS ziyaret sayılarını taklit eder).
2. **Value Head (Değer Çıktısı):** Mevcut tahta durumundan oyunun kimin kazanacağına dair [-1, 1] aralığında skaler bir tahmin.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual # Skip Connection
        return F.relu(out)

class PolicyValueNet(nn.Module):
    def __init__(self, in_channels=2, num_res_blocks=5, channels=64):
        super().__init__()
        # Giriş Katmanı (Shared Representation)
        self.conv_in = nn.Conv2d(in_channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn_in = nn.BatchNorm2d(channels)

        # Deeper ResNet (4-6 blocks recommended, using 5)
        self.res_blocks = nn.ModuleList([ResBlock(channels) for _ in range(num_res_blocks)])

        # Policy Head
        self.policy_conv = nn.Conv2d(channels, 2, kernel_size=1, bias=False)
        self.policy_bn = nn.BatchNorm2d(2)
        self.policy_fc = nn.Linear(2 * 5 * 5, 25)

        # Value Head
        self.value_conv = nn.Conv2d(channels, 1, kernel_size=1, bias=False)
        self.value_bn = nn.BatchNorm2d(1)
        self.value_fc1 = nn.Linear(1 * 5 * 5, 32)
        self.value_fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.relu(self.bn_in(self.conv_in(x)))

        for block in self.res_blocks:
            x = block(x)

        # Policy Hesaplaması
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        policy_logits = self.policy_fc(p)

        # Value Hesaplaması
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))

        return policy_logits, value

# ==========================================
# MCTS IMPLEMENTATION (INFERENCE ONLY)
# ==========================================

class MCTSNode:
    def __init__(self, parent=None, prior_prob=1.0):
        self.parent = parent
        self.children = {} # action: Node
        self.visit_count = 0
        self.value_sum = 0.0
        self.prior_prob = prior_prob

    @property
    def q_value(self):
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count

    def expand(self, action_probs):
        """Gelen policy olasılıklarına göre çocuk düğümleri yaratır."""
        for action, prob in action_probs:
            if action not in self.children:
                self.children[action] = MCTSNode(parent=self, prior_prob=prob)

    def is_expanded(self):
        return len(self.children) > 0

class MCTS:
    def __init__(self, model, c_puct=1.5, num_simulations=50):
        self.model = model
        self.c_puct = c_puct
        self.num_simulations = num_simulations
        self.model.eval()

    def search(self, root_state, legal_moves):
        """
        root_state: (1, 2, 5, 5) tensor
        legal_moves: list of valid action integers
        Döndürdüğü: En iyi action
        """
        root = MCTSNode()

        # İlk düğümü genişlet
        with torch.no_grad():
            state_tensor = root_state.to(next(self.model.parameters()).device)
            policy_logits, _ = self.model(state_tensor)
            # Sadece yasal hamlelere softmax
            mask = torch.zeros(25, device=policy_logits.device)
            mask[legal_moves] = 1.0
            policy_logits = policy_logits.masked_fill(mask == 0, -1e4)
            probs = F.softmax(policy_logits, dim=1).cpu().numpy()[0]

        action_probs = [(a, probs[a]) for a in legal_moves]
        root.expand(action_probs)

        for _ in range(self.num_simulations):
            node = root
            # Seçim aşaması (Selection)
            # Not: Gerçek bir oyunda state'in güncellenerek inilmesi gerekir.
            # Burada konsept olarak temel ağaç inme mantığı gösterilmiştir.
            search_path = [node]
            while node.is_expanded():
                action, node = self._select_child(node)
                search_path.append(node)

            # Genişletme & Değerlendirme (Expansion & Evaluation)
            # Normalde simüle edilen oyun state'i burada ağa verilir.
            # Biz basitleştirilmiş bir mock value döndürüyoruz (Ajan entegrasyonu için).
            leaf_value = 0.0 # Placeholder: Ağdan dönen Value olmalı.

            # Geri yayılım (Backpropagation)
            self._backpropagate(search_path, leaf_value)

        # En çok ziyaret edilen hamleyi seç
        best_action = max(root.children.items(), key=lambda item: item[1].visit_count)[0]
        return best_action

    def _select_child(self, node):
        best_score = -float('inf')
        best_action = -1
        best_child = None

        for action, child in node.children.items():
            # PUCT Formula
            u = self.c_puct * child.prior_prob * math.sqrt(node.visit_count) / (1 + child.visit_count)
            score = child.q_value + u
            if score > best_score:
                best_score = score
                best_action = action
                best_child = child

        return best_action, best_child

    def _backpropagate(self, search_path, value):
        # Sırayla root'a kadar değerleri güncelle
        for node in reversed(search_path):
            node.value_sum += value
            node.visit_count += 1
            value = -value # Sıra karşı oyuncuya geçiyorsa tersine çevrilmeli


## 3. Veri Hattı (Data Pipeline) ve RAM Optimizasyonu

Eğitim darboğazlarını (I/O Bottlenecks) engellemek için, 2.4 GB'lık verimsiz JSON veri seti yapısı sıkıştırılmış PyTorch (`.pt`) formatına dönüştürülmüştür.
Modelin GPU'da veri beklemesini önlemek adına tüm veri **doğrudan RAM'e (In-Memory)** yüklenir. Ayrıca `DataLoader` seviyesinde `pin_memory=True` ve çoklu işlem (multiprocessing) kullanılmıştır.

In [ ]:
import random

def build_combined_dataset(json_paths, pt_path):
    """
    Birden fazla JSON veri setini birleştirip yüksek performanslı binary formata (.pt) dönüştürür.

    Neden Farklı Veri Setlerini Karıştırıyoruz?
    Karmaşık (Minimax) ve rastgele (Random) ajan oyunlarını birleştirmek, modelin hem
    yüksek seviye stratejileri hem de temel kuralları öğrenmesini sağlar. Minimax verisi
    optimum hamleleri gösterirken, rastgele veriler modelin beklenmedik durumlara karşı
    dayanıklılığını artırır. AlphaZero gibi kendi kendine oynayan (self-play) sistemlerde de
    ajan başlangıçta rastgele hamleler yapıp zamanla ustalaşarak benzer bir veri çeşitliliği yaratır.
    """
    if os.path.exists(pt_path):
        print(f"[BİLGİ] Optimize format {pt_path} bulundu. Dönüşüm atlanıyor.")
        return

    print("[İŞLEM] JSON dosyaları okunuyor ve birleştiriliyor...")
    combined_data = []
    for json_path in json_paths:
        print(f"  -> Okunuyor: {json_path}")
        with open(json_path, 'r') as f:
            data = json.load(f)
            combined_data.extend(data)

    print("[İŞLEM] Veriler karıştırılıyor (Shuffling)...")
    random.shuffle(combined_data)

    print(f"[İŞLEM] Toplam {len(combined_data)} örnek Tensor formatına dönüştürülüyor...")
    num_samples = len(combined_data)
    states = torch.empty((num_samples, 2, 5, 5), dtype=torch.float32)
    policies = torch.empty((num_samples, 25), dtype=torch.float32)
    values = torch.empty((num_samples, 1), dtype=torch.float32)
    masks = torch.empty((num_samples, 25), dtype=torch.float32)

    for i, item in enumerate(combined_data):
        states[i] = torch.from_numpy(np.array(item['encoded_state'], dtype=np.float32))
        policies[i] = torch.from_numpy(np.array(item['policy_target'], dtype=np.float32))
        values[i] = torch.from_numpy(np.array(item['value_target'], dtype=np.float32))
        masks[i] = torch.from_numpy(np.array(item['legal_action_mask'], dtype=np.float32))

    assert states.shape[1:] == (2, 5, 5), f"State shape yanlış: {states.shape}"
    assert policies.shape[1] == 25, f"Policy shape yanlış: {policies.shape}"

    torch.save({'states': states, 'policies': policies, 'values': values, 'masks': masks}, pt_path)
    print("[BAŞARILI] Birleştirme ve dönüşüm tamamlandı!")

class InMemoryDataset(Dataset):
    """I/O Darboğazını sıfırlayan, veriyi tamamen bellekte (RAM) tutan özel sınıf."""
    def __init__(self, pt_path):
        print("[İŞLEM] Veri RAM'e yükleniyor...")
        data = torch.load(pt_path, weights_only=True)
        self.states = data['states']
        self.policies = data['policies']
        self.values = data['values']
        self.masks = data['masks']
        print(f"[BAŞARILI] Toplam {len(self.states)} örnek belleğe yüklendi.")

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return self.states[idx], self.policies[idx], self.values[idx], self.masks[idx]


## 4. Eğitim Döngüsü (Mixed Precision Training)

Eğitim aşamasında performansı katlamak için **PyTorch Automatic Mixed Precision (AMP)** kullanılmıştır. A100 GPU, FP32 yerine FP16 hesaplamaları yaparak eğitim süresini yarıya indirir. Ayrıca geçersiz hamlelerin olasılıkları `-1e4` değeri ile maskelenerek ağın sadece yasal hamlelere odaklanması sağlanmıştır.

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# 1. Veri Hazırlığı (FIXED DATASET KULLAN!)
data_dir = "/content/drive/MyDrive/dataset final/"

JSON_PATHS = [
    os.path.join(data_dir, "mh_40k_fixed_20260324_152733.json"),
    os.path.join(data_dir, "hh_30k_fixed_20260324_154021.json"),
    os.path.join(data_dir, "hr_20k_fixed_20260324_154041.json"),
    os.path.join(data_dir, "rr_10k_fixed_20260324_154054.json")
]

PT_PATH = os.path.join(data_dir, "final_dataset_fixed.pt")

# Dataset oluştur (bir kez çalıştır yeter)
if not os.path.exists(PT_PATH):
    build_combined_dataset(JSON_PATHS, PT_PATH)

dataset = InMemoryDataset(PT_PATH)

# 2. DataLoader (STABLE AYARLAR)
batch_size = 1024  # 🔥 4096 yerine daha stabil
num_workers = min(os.cpu_count(), 8)

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True
)

# 3. Model + Optimizer (OPTIMIZED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PolicyValueNet().to(device)

if torch.__version__ >= '2.0.0':
    model = torch.compile(model)

# 🔥 LR + REGULARIZATION FIX
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-4,          # ⬅️ düşürdük
    weight_decay=5e-4 # ⬅️ artırdık
)

# 🔥 LR SCHEDULER (çok önemli)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=25
)

scaler = torch.amp.GradScaler()

policy_criterion = nn.CrossEntropyLoss()
value_criterion = nn.MSELoss()
value_loss_weight = 0.1
entropy_weight = 0.02  # 🔥 ENTROPY REGULARIZATION KATSAYISI (0.01 -> 0.02)

# 🔥 EPOCH ARTTIR
num_epochs = 25

print("\n--- EĞİTİM BAŞLIYOR ---")
start_train = time.time()

# 4. Eğitim Döngüsü
for epoch in range(num_epochs):
    epoch_start = time.time()
    model.train()

    train_loss_sum = 0.0

    for states, policies, values, masks in train_loader:

        states = states.to(device, non_blocking=True)
        policies = policies.to(device, non_blocking=True)
        values = values.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type='cuda'):
            policy_logits, pred_values = model(states)

            # illegal move mask applied BEFORE loss
            policy_logits = policy_logits.masked_fill(masks == 0, -1e4)

            target_actions = torch.argmax(policies, dim=1)

            loss_p = policy_criterion(policy_logits, target_actions)
            loss_v = value_criterion(pred_values.squeeze(), values.squeeze())

            # Numerically stable entropy from softmax
            probs = F.softmax(policy_logits, dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=1).mean()

            # Updated Final Loss formula
            total_loss = loss_p + value_loss_weight * loss_v - entropy_weight * entropy

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += total_loss.item()

    train_loss = train_loss_sum / len(train_loader)

    # --- VALIDATION ---
    model.eval()

    val_loss_sum = 0.0
    top1_correct = 0
    top3_correct = 0
    mae_sum = 0.0
    entropy_sum = 0.0
    total_samples = 0

    with torch.no_grad():
        for states, policies, values, masks in val_loader:

            states = states.to(device, non_blocking=True)
            policies = policies.to(device, non_blocking=True)
            values = values.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)


            with torch.amp.autocast(device_type='cuda'):
                policy_logits, pred_values = model(states)

                # Mask fallback
                safe_masks = masks.clone()

                mask_sum = masks.sum(dim=1)
                invalid_rows = (mask_sum == 0)
                if invalid_rows.any():
                    safe_masks[invalid_rows] = 1   # fallback

                policy_logits = policy_logits.masked_fill(safe_masks == 0, -1e4)

                target_actions = torch.argmax(policies, dim=1)

                loss_p = policy_criterion(policy_logits, target_actions)
                loss_v = value_criterion(pred_values.squeeze(), values.squeeze())

                # Validation does NOT include entropy bonus in val_loss
                val_loss_sum += (loss_p + value_loss_weight * loss_v).item()

            # metrics
            _, top3_preds = torch.topk(policy_logits, 3, dim=1)

            top1_correct += (top3_preds[:, 0] == target_actions).sum().item()
            top3_correct += (top3_preds == target_actions.unsqueeze(1)).any(dim=1).sum().item()

            mae_sum += torch.abs(pred_values.squeeze() - values.squeeze()).sum().item()

            # ENTROPY FIX (en stabil)
            entropy = torch.distributions.Categorical(
                logits=policy_logits.float()
            ).entropy()

            entropy_sum += entropy.sum().item()

            total_samples += states.size(0)

    val_loss = val_loss_sum / len(val_loader)
    val_top1 = 100 * top1_correct / total_samples
    val_top3 = 100 * top3_correct / total_samples
    val_mae = mae_sum / total_samples
    val_entropy = entropy_sum / total_samples
    scheduler.step()

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.3f} | "
        f"Val Loss: {val_loss:.3f} | "
        f"Top-1: %{val_top1:.1f} | "
        f"Top-3: %{val_top3:.1f} | "
        f"MAE: {val_mae:.3f} | "
        f"Entropy: {val_entropy:.2f} | "
        f"Süre: {time.time() - epoch_start:.1f}s"
    )

print(f"\n--- EĞİTİM BİTTİ ({time.time() - start_train:.1f} saniye) ---")

[İŞLEM] Veri RAM'e yükleniyor...
[BAŞARILI] Toplam 1600000 örnek belleğe yüklendi.

--- EĞİTİM BAŞLIYOR ---
Epoch 01/25 | Train Loss: 0.726 | Val Loss: 0.685 | Top-1: %78.2 | Top-3: %83.6 | MAE: 0.246 | Entropy: 0.68 | Süre: 23.5s
Epoch 02/25 | Train Loss: 0.668 | Val Loss: 2.363 | Top-1: %55.6 | Top-3: %64.8 | MAE: 0.418 | Entropy: 2.50 | Süre: 17.1s
Epoch 03/25 | Train Loss: 0.666 | Val Loss: 0.680 | Top-1: %78.4 | Top-3: %83.8 | MAE: 0.238 | Entropy: 0.67 | Süre: 17.7s
Epoch 04/25 | Train Loss: 0.664 | Val Loss: 0.679 | Top-1: %78.5 | Top-3: %83.7 | MAE: 0.254 | Entropy: 0.66 | Süre: 18.5s
Epoch 05/25 | Train Loss: 0.663 | Val Loss: 0.677 | Top-1: %78.4 | Top-3: %83.6 | MAE: 0.245 | Entropy: 0.67 | Süre: 18.0s
Epoch 06/25 | Train Loss: 0.662 | Val Loss: 0.676 | Top-1: %78.5 | Top-3: %83.7 | MAE: 0.235 | Entropy: 0.66 | Süre: 17.8s
Epoch 07/25 | Train Loss: 0.661 | Val Loss: 0.677 | Top-1: %78.4 | Top-3: %83.7 | MAE: 0.241 | Entropy: 0.66 | Süre: 18.5s
Epoch 08/25 | Train Loss: 0.660

In [ ]:
# ==============================
# GAME LOGIC (5x5 board)
# ==============================

import numpy as np

class SimpleGame:
    def __init__(self):
        self.board = np.zeros((5,5), dtype=int)
        self.current_player = 1  # 1 or -1

    def clone(self):
        new_game = SimpleGame()
        new_game.board = self.board.copy()
        new_game.current_player = self.current_player
        return new_game

    def get_legal_moves(self):
        return [i for i in range(25) if self.board[i//5, i%5] == 0]

    def apply_move(self, action):
        r, c = action // 5, action % 5
        self.board[r, c] = self.current_player
        self.current_player *= -1

    def is_game_over(self):
        return len(self.get_legal_moves()) == 0

    def get_winner(self):
        return np.sum(self.board)  # basit skor (placeholder)

    def encode(self):
        state = np.zeros((2,5,5), dtype=np.float32)
        state[0] = (self.board == 1)
        state[1] = (self.board == -1)
        return state

In [ ]:
# ==========================================
# ALPHAZERO-STYLE MCTS (UPGRADED)
# ==========================================

class MCTSNode:
    def __init__(self, parent=None, prior=0.0):
        self.parent = parent
        self.children = {}
        self.visit_count = 0
        self.value_sum = 0.0
        self.prior = prior

    @property
    def q_value(self):
        return 0 if self.visit_count == 0 else self.value_sum / self.visit_count

    def expand(self, action_probs):
        for action, prob in action_probs:
            if action not in self.children:
                self.children[action] = MCTSNode(parent=self, prior=prob)

    def is_expanded(self):
        return len(self.children) > 0


class MCTS:
    def __init__(self, model, c_puct=1.5, num_simulations=50):
        self.model = model
        self.c_puct = c_puct
        self.num_simulations = num_simulations
        self.root = None

    def search(self, game):
        if self.root is None:
            self.root = MCTSNode()

        for _ in range(self.num_simulations):
            game_copy = game.clone()
            self._simulate(game_copy, self.root)

        # best move
        action = max(self.root.children.items(), key=lambda x: x[1].visit_count)[0]

        # 🌟 TREE REUSE
        self.root = self.root.children[action]
        self.root.parent = None

        return action

    def _simulate(self, game, node):

        if game.is_game_over():
            value = game.get_winner()
            self._backpropagate(node, value)
            return

        if not node.is_expanded():
            # evaluate with network
            state = torch.tensor(game.encode()).unsqueeze(0).to(device)

            with torch.no_grad():
                policy_logits, value = self.model(state)

            legal_moves = game.get_legal_moves()

            mask = torch.zeros(25, device=device)
            mask[legal_moves] = 1

            policy_logits = policy_logits.masked_fill(mask == 0, -1e4)
            probs = F.softmax(policy_logits, dim=1).cpu().numpy()[0]

            node.expand([(a, probs[a]) for a in legal_moves])

            self._backpropagate(node, value.item())
            return

        # SELECT
        action, child = self._select(node)

        game.apply_move(action)

        self._simulate(game, child)

    def _select(self, node):
        best_score = -float('inf')
        best_action = None
        best_child = None

        for action, child in node.children.items():
            u = self.c_puct * child.prior * math.sqrt(node.visit_count + 1) / (1 + child.visit_count)
            score = child.q_value + u

            if score > best_score:
                best_score = score
                best_action = action
                best_child = child

        return best_action, best_child

    def _backpropagate(self, node, value):
        while node is not None:
            node.value_sum += value
            node.visit_count += 1
            value = -value
            node = node.parent

In [ ]:
# ==========================================
# AI vs RANDOM TEST
# ==========================================

def play_game(model, mcts, verbose=False):
    game = SimpleGame()

    while not game.is_game_over():

        if game.current_player == 1:
            # AI
            action = mcts.search(game)
        else:
            # RANDOM
            action = np.random.choice(game.get_legal_moves())

        game.apply_move(action)

        if verbose:
            print(game.board)
            print("----")

    return game.get_winner()


def evaluate(model, num_games=20):
    mcts = MCTS(model)

    results = []

    for i in range(num_games):
        winner = play_game(model, mcts)
        results.append(winner)

    results = np.array(results)

    ai_wins = np.sum(results > 0)
    random_wins = np.sum(results < 0)
    draws = np.sum(results == 0)

    print("\n===== RESULTS =====")
    print(f"AI Wins: {ai_wins}")
    print(f"Random Wins: {random_wins}")
    print(f"Draws: {draws}")
    print(f"Win Rate: {ai_wins / num_games * 100:.2f}%")

In [ ]:
model.eval()
evaluate(model, num_games=30)


===== RESULTS =====
AI Wins: 30
Random Wins: 0
Draws: 0
Win Rate: 100.00%


In [ ]:
# ==========================================
# MCTS KULLANIM ÖRNEĞİ (INFERENCE / GAMEPLAY)
# ==========================================
import torch

print("\n--- MCTS ÖRNEK KULLANIMI ---")

# Eğer model henüz tanımlanmadıysa (önceki hücreler çalıştırılmadıysa) örnek bir model oluşturalım
if 'model' not in globals():
    print("Model tanımlanmamış, örnek kullanım için yeni bir PolicyValueNet oluşturuluyor...")
    model = PolicyValueNet()

# 1. Eğitilmiş modeli değerlendirme moduna (eval) alıyoruz
model.eval()

# 2. MCTS motorunu başlatıyoruz
# c_puct: Keşif (exploration) katsayısı (genelde 1.0 - 1.5 arası iyidir)
# num_simulations: Her hamle seçimi öncesi yapılacak ağaç araması (50 idealdir)
mcts = MCTS(model=model, c_puct=1.5, num_simulations=50)

# 3. Örnek bir oyun durumu (Board State) oluşturalım
# Format: (Batch=1, Channels=2, Height=5, Width=5)
dummy_state = torch.zeros((1, 2, 5, 5), dtype=torch.float32)

# 4. Geçerli hamleleri (Legal Moves) belirleyelim
# Tahtadaki kurallara göre o an oynanabilecek hamlelerin indeksleri (0-24 arası)
# Örnek olarak 5 farklı hamlenin yasal olduğunu varsayalım:
sample_legal_moves = [0, 1, 5, 12, 24]

# 5. MCTS kullanarak en iyi hamleyi bulalım
# Bu işlem, verdiğimiz state üzerinden 'num_simulations' kadar arama yapar.
game = SimpleGame()

# örnek state doldur (isteğe bağlı)
game.board[0,0] = 1
game.board[0,1] = -1

best_action = mcts.search(game)

print(f"Mevcut durum için geçerli hamleler: {sample_legal_moves}")
print(f"MCTS tarafından seçilen en iyi hamle (Action ID): {best_action}")


--- MCTS ÖRNEK KULLANIMI ---
Mevcut durum için geçerli hamleler: [0, 1, 5, 12, 24]
MCTS tarafından seçilen en iyi hamle (Action ID): 12


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/model.pth")

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/model.pth"

torch.save(model.state_dict(), SAVE_PATH)

print("Kaydedildi:", SAVE_PATH)

Kaydedildi: /content/drive/MyDrive/model.pth
